# 16장 · 갈림길의 정원

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [16장 · 갈림길의 정원](https://grow.minds.kr/textbooks/css-methods/causal/book/ch16-갈림길의-정원.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 갈림길의 정원
효과가 확실히 없는 세계(조건 딱지를 뒤섞음)에서, 분석 갈림길을 다 걸으면 '유의'가 수확됩니다.

In [ ]:
exp = pd.read_csv("data/journey_exp.csv")
rng = np.random.default_rng(73)
null_cond = rng.permutation(exp.cond.values)
d = exp.assign(cond=null_cond); dc = d[d.attn_1==1]
tt = stats.ttest_ind(dc[dc.cond==1].mil, dc[dc.cond==0].mil)
print("사전 지정 주 분석 p =", round(tt.pvalue,3))   # 0.346 (정직한 답)

## 3. 60갈래를 다 걸으면
결과 2 × 표본 3 × 하위집단 5 × 공변인 2 = 60. 하나하나는 합리적인데, 다 걸어 예쁜 것만 고르면?

In [ ]:
def forks(df, cond):
    out=[]; d=df.copy(); d["cond"]=cond
    for oc in ("mil","flr"):
        for excl in ("none","attn","trim"):
            e=d
            if excl=="attn": e=d[d.attn_1==1]
            elif excl=="trim":
                z=(d[oc]-d[oc].mean())/d[oc].std(ddof=1); e=d[np.abs(z)<=2.5]
            for sub in ("all","m","f","young","old"):
                s=e
                if sub=="m": s=e[e.gender==1]
                elif sub=="f": s=e[e.gender==2]
                elif sub=="young": s=e[e.age<=e.age.median()]
                elif sub=="old": s=e[e.age>e.age.median()]
                for cov in (False, True):        # 공변인(기저 mil_t1) 넣기/빼기
                    if cov:
                        _,_,p,_ = ols(s[oc].values, [s.cond.values, s.mil_t1.values]); out.append(p[1])
                    else:
                        out.append(stats.ttest_ind(s[s.cond==1][oc], s[s.cond==0][oc]).pvalue)
    return out
ps = forks(exp, null_cond)
print("갈림길:", len(ps), " p<.05 수확:", sum(p<.05 for p in ps), " 최소 p:", round(min(ps),3))
# 사전 지정 하나(0.346)는 정직하지만, 60을 다 걸으면 그럴듯한 유의가 여럿 나온다.

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.